# Notebook 2 — Real databases (EXIOBASE)

In Notebook 1 we learned the mechanics on a toy database. Now we move to **real,
multi-regional data**. EXIOBASE is a global Multi-Regional database covering ~49
regions and ~163–200 sectors/products, available in two flavours:

- a **hybrid** version, where physical flows are measured in their *natural*
  units (tonnes, TJ, m³, …) — a Supply-Use Table (HSUT);
- a **monetary** version, where everything is in money (M€) — available as both
  SUT and symmetric IOT.

This notebook does four things:

1. parse the **hybrid HSUT** and show its defining feature — *mixed units*;
2. free the memory and switch to the **monetary IOT**;
3. **aggregate** that huge table down to a compact model (≈7 sectors,
   6 macro-regions) keeping only value added, CO₂ and water;
4. **visualize, export and save** the result for Notebook 3.

> ⚠️ **Heavy data.** The EXIOBASE bundles are several GB. You download them once
> to a local folder and then point MARIO at that folder. These cells are meant to
> be run on your own machine — not in a shared classroom session. Edit the paths
> in the next cell before running.

In [ ]:
import mario
import gc

mario.set_log_verbosity("info")   # we WANT to see the parser/aggregation logs here
print("MARIO version:", mario.__version__)

# ---------------------------------------------------------------------------
# EDIT THESE PATHS to point at your local EXIOBASE folders.
# ---------------------------------------------------------------------------
HYBRID_PATH   = r"D:/EXIOBASE/3.3.18"          # extracted hybrid release (HSUT/HIOT)
MONETARY_PATH = r"D:/EXIOBASE/3.10.2"          # destination/cache for the monetary IOT
MONETARY_YEAR = 2022

# Where this notebook writes its outputs (exports + the parquet snapshot for NB3).
OUT_DIR = "outputs"
import os
os.makedirs(OUT_DIR, exist_ok=True)

## Part 1 — Hybrid EXIOBASE (HSUT)

The hybrid release MARIO currently supports is **v3.3.18**
([Zenodo](https://doi.org/10.5281/zenodo.7244919)). Its key feature is that each
commodity is measured in its own physical unit, so the table is **not additive**
across rows — exactly the point we want to show.

### 1.1 Download (run once)

`download_hybrid_exiobase(...)` fetches the supported bundle into `path`. It is a
large, slow, one-time operation; skip it if you already have the files.

In [ ]:
# Run once. Comment out after the files are on disk.
info = mario.download_hybrid_exiobase(path=HYBRID_PATH)
info

### 1.2 Parse the hybrid Supply-Use Table

We parse the HSUT with `unit="Hybrid"`. We keep the extension selection explicit
and light (`["resource", "Emiss"]`) so the parse stays fast; pass `"all"` to load
every extension group.

In [ ]:
db_hsut = mario.parse_exiobase(
    path=HYBRID_PATH,
    table="SUT",
    unit="Hybrid",
    extensions=["resource", "Emiss"],
)
db_hsut

### 1.3 The peculiarity: mixed units

In a monetary table every cell is in money and rows can be summed. In the hybrid
table, units **differ from one commodity to the next**. MARIO keeps a dedicated
`units` dictionary per set — always check it before doing arithmetic on a hybrid
table.

In [ ]:
# Each commodity carries its own physical unit (tonnes, TJ, m³, ...)
db_hsut.units["Commodity"].head(15)

In [ ]:
# How many distinct units appear across commodities?
db_hsut.units["Commodity"]["unit"].value_counts()

In [ ]:
# Concrete consequence: summing a physical-use column mixes tonnes, TJ, m³...
# which is meaningless. Always group by unit first. Example — the use structure
# of one activity, split by the unit of the used commodity:
use_col = db_hsut.U.iloc[:, 0].to_frame("use")
use_col["unit"] = db_hsut.units["Commodity"]["unit"].reindex(
    use_col.index.get_level_values("Item")
).values
use_col.groupby("unit")["use"].sum()

> 🔑 **Takeaway.** Hybrid tables are ideal for *physical* footprint analysis
> (energy, materials, emissions in their own units) but you cannot treat them like
> a monetary table. For an economy-wide, additive model we switch to the monetary
> version below.

### 1.4 Free the memory

Real MRIO objects are large. Before loading the next one, delete the hybrid
database and force garbage collection so we don't keep two multi-GB tables in RAM
at the same time.

In [ ]:
del db_hsut
gc.collect()
print("Hybrid database released.")

## Part 2 — Monetary IOT

Now the monetary symmetric **Input-Output Table**. Recent monetary releases are
IOT-only; we use one of them (here 3.10.2) and the industry-by-industry system
(`ixi`).

### 2.1 Download (run once)

In [ ]:
# Run once. Comment out after the files are on disk.
mario.download_exiobase3(
    path=MONETARY_PATH,
    years=[MONETARY_YEAR],
    table="IOT",
    system="ixi",
    version="3.10.2",
)

### 2.2 Parse the monetary IOT

Point `path` at the extracted `IOT_<year>_ixi` folder. Parsing brings in ~163
sectors, 49 regions, the value-added block and several hundred satellite
accounts.

In [ ]:
db = mario.parse_exiobase(
    path=os.path.join(MONETARY_PATH, f"IOT_{MONETARY_YEAR}_ixi"),
    table="IOT",
    unit="Monetary",
)
db

In [ ]:
print("Regions:", len(db.regions))
print("Sectors:", len(db.sectors))
print("Factors of production (value added):", len(db.get_index("Factor of production")))
print("Satellite accounts:", len(db.get_index("Satellite account")))

## Part 3 — Aggregate to a compact model

163 sectors × 49 regions is too much to teach (or shock) clearly. We aggregate
to a small, interpretable model:

- **7 sectors:** Agriculture, Renewable electricity, Non-renewable electricity,
  Electrical equipment, Rare materials, Manufacturing, Services;
- **6 macro-regions:** Africa, USA, Europe, China, Rest of Asia, Rest of the World;
- **value added:** kept (all factor rows);
- **satellite accounts:** keep only **CO₂** and **Water**, drop the rest.

MARIO aggregates from an **Excel workbook**: one sheet per set, each with an
`Aggregation` column where you write, for every original label, the name of the
group it belongs to. The workflow is meant to be done **by hand**:

1. generate the **empty template** with `get_aggregation_excel(...)`;
2. **open it in Excel and fill the `Aggregation` column** following the grouping
   below;
3. apply it with `db.aggregate("...your_filled_file.xlsx")`.

### 3.1 Generate the empty template

This writes one sheet per set, each listing the original labels with an empty
`Aggregation` column next to them.

In [ ]:
AGG_PATH = os.path.join(OUT_DIR, "aggregation_exio.xlsx")
db.get_aggregation_excel(path=AGG_PATH, overwrite=True)
print("Empty template written to:", AGG_PATH)

### 3.2 Fill in the `Aggregation` column

Open `outputs/aggregation_exio.xlsx` and fill each sheet as described here. (In a
live session you would prepare this file in advance and hand it out already
filled.)

**`Region` sheet** — map each EXIOBASE country code to one of 6 macro-regions:

| Write in `Aggregation` | For these region codes |
|---|---|
| `USA` | US |
| `China` | CN |
| `Europe` | AT, BE, BG, CY, CZ, DE, DK, EE, ES, FI, FR, GR, HR, HU, IE, IT, LT, LU, LV, MT, NL, PL, PT, RO, SE, SI, SK, GB, CH, NO, WE |
| `Africa` | ZA, WF |
| `Rest of Asia` | JP, KR, IN, ID, TW, TR, WA, WM |
| `Rest of the World` | CA, BR, MX, RU, AU, WL |

**`Sector` sheet** — map each of the ~163 sectors to one of 7 groups:

| Write in `Aggregation` | Which EXIOBASE sectors |
|---|---|
| `Renewable electricity` | *Production of electricity by* hydro / wind / solar / biomass / tide / geothermal |
| `Non-renewable electricity` | *Production of electricity by* coal / gas / nuclear / petroleum, and *Production of electricity nec* |
| `Electrical equipment` | electrical machinery, office machinery & computers, radio/TV/communication equipment, medical/precision/optical instruments |
| `Rare materials` | mining **and** production of precious metals, copper, nickel, aluminium, lead/zinc/tin and other non-ferrous metals |
| `Agriculture` | cultivation, farming, forestry, fishing, raw milk, animal products, wool, manure |
| `Services` | trade, transport, retail/wholesale, hotels/restaurants, telecom, finance/insurance, real estate, renting, IT & R&D, public administration, education, health, recreation, electricity grid (transmission/distribution), gas/steam/water utilities, waste treatment & recycling, vehicle sale & repair |
| `Manufacturing` | everything else — fossil extraction, refining, chemicals, plastics, basic metals & steel, cement, machinery, vehicles, food processing, construction, … |

**`Satellite account` sheet** — we keep only two impacts and drop the hundreds of
others:

| Write in `Aggregation` | Which accounts |
|---|---|
| `CO2` | any account whose name contains *CO2* or *Carbon dioxide* |
| `Water` | any account whose name contains *Water* |
| `unused` | **everything else** (labels mapped to `unused` are dropped) |

**`Factor of production` and `Consumption category` sheets** — **leave the
`Aggregation` column blank**. With `ignore_nan=True`, blank labels are left
untouched, so value added and the demand categories are kept as they are.

### 3.3 Apply the aggregation

Once the workbook is filled and saved, `db.aggregate(...)` reads it and collapses
every level at once. Labels mapped to `"unused"` (all the satellite accounts
except CO₂ and Water) are **dropped**.

In [ ]:
db.aggregate(AGG_PATH, ignore_nan=True)   # drop=['unused'] is the default

print(db)
print("\nSectors:", db.sectors)
print("Regions:", db.regions)
print("Satellite accounts:", db.get_index("Satellite account"))

## Part 4 — Visualize the aggregated table

Now that the table is small, the matrices are readable. Let's compute the core
blocks and visualize the intermediate transactions `Z`.

In [ ]:
db.calc_all(["X", "Z", "z", "Y", "v", "e"])
db.Z

In [ ]:
# Bar chart of the transaction matrix Z (input per unit of output)
fig = db.plot(matrix="Z", preset="overview", path=False, auto_open=False)
fig

In [ ]:
# satellite accounts (the factors that appear in Item_from), straight from MARIO
factors = db.get_index("Satellite account")   # list of satellite account names

# numerator unit per satellite account, from MARIO's own units
ext_units = db.units["Satellite account"]     # DataFrame indexed by factor, "unit" column

# denominator: sector output unit (X)
den = db.units["Sector"].iloc[0, 0]           # e.g. "M€"

# factor -> "numerator / denominator", built only from the real satellite accounts
unit_map = {f: f"{ext_units.loc[f, 'unit']} / {den}" for f in factors}

In [ ]:
# CO2 footprint intensity by sector (color = region)
fig = db.plot(
    matrix="f",
    kind="bar",
    preset=None,
    x="Item_to",
    color="Region",
    facet_col="Item_from",
    agg="sum",
    barmode="group",      # grouped bars instead of stacked
    path=False,
    auto_open=False,
)

fig.update_yaxes(matches=None, showticklabels=True)

# attach each satellite account's unit to its facet title
for ann in fig.layout.annotations:
    if "Item_from=" in ann.text:
        f = ann.text.split("Item_from=")[-1]
        ann.text = f"{f} [{unit_map.get(f, '')}]"

fig.show()

## Part 5 — Export and save for Notebook 3

We export the database **in coefficients** (handy to inspect the technical
structure in Excel) and save a compact **parquet** snapshot. Notebook 3 starts by
loading that snapshot, so it never has to re-parse the multi-GB raw data.

In [ ]:
# Coefficient export to Excel (z, v, e + the absolute Y and X)
db.to_excel(
    path=os.path.join(OUT_DIR, "exio_aggregated_coefficients.xlsx"),
    flows=False,
    coefficients=True,
)
print("Coefficient workbook written.")

In [ ]:
# Parquet snapshot (flat layout, both flows and coefficients) for the round-trip
PARQUET_PATH = os.path.join(OUT_DIR, "exio_aggregated_parquet")
db.to_parquet(
    path=PARQUET_PATH,
    flows=True,
    coefficients=True,
    flat=True,
)
print("Parquet snapshot written to:", PARQUET_PATH)

In [ ]:
# Verify the round-trip: parse the parquet back and check it matches.
check = mario.parse_from_parquet(
    PARQUET_PATH,
    table="IOT",
    mode="flows",
    new_scenario="baseline",
    flat=True,
)
print(check)
print("\nSectors match:", check.sectors == db.sectors)
print("Regions match:", check.regions == db.regions)

✅ The aggregated, real-data model is now saved.

**Notebook 3** picks it up from here: we add a brand-new sector (an AI
"hyperscaler"), apply a final-demand shock, and build the AI-substitution
scenario — then visualize the winners and losers.